## Fluxo para gerar Análise estatística e métricas

In [ ]:
# !pip install matplotlib
!python model_results_stat_analysis.py

In [ ]:
!pip install umap-learn matplotlib numpy

In [ ]:
import pandas as pd

# caminho do arquivo
file_path = r"C:codes\results\classical_models\classical_models_test_predictions.csv"

# leitura
df = pd.read_csv(file_path)

# garante consistência
df["split"] = df["split"].astype(str).str.lower().str.strip()

# filtra somente test
df_test = df[df["split"] == "test"].copy()

# marca erro
df_test["is_error"] = (df_test["true_label_id"] != df_test["pred_label_id"]).astype(int)

# mantém só os erros
df_errors = df_test[df_test["is_error"] == 1].copy()

# agrupa por dataset_id + idx
df_summary = (
    df_errors
    .groupby(["dataset_id", "idx"], as_index=False)
    .agg(
        error_count=("is_error", "sum"),
        n_models_error=("model", "nunique"),
        models_error=("model", lambda x: sorted(set(x))),
        embedding_models_error=("embedding_model", lambda x: sorted(set(x))),
        classifiers_error=("classifier", lambda x: sorted(set(x))),
        true_label_id=("true_label_id", "first"),
    )
)

# ordena do mais errado para o menos errado
df_summary = df_summary.sort_values(
    by=["dataset_id", "error_count", "n_models_error", "idx"],
    ascending=[True, False, False, True]
).copy()

# pega top 2 idx por dataset_id
df_top2 = (
    df_summary
    .groupby("dataset_id", group_keys=False)
    .head(2)
    .reset_index(drop=True)
)

# print organizado
for dataset_id, group in df_top2.groupby("dataset_id"):
    print("=" * 100)
    print(f"dataset_id: {dataset_id}")
    
    for i, row in enumerate(group.itertuples(index=False), start=1):
        print(f"\nTop {i}")
        print(f"idx: {row.idx}")
        print(f"true_label_id: {row.true_label_id}")
        print(f"error_count: {row.error_count}")
        print(f"n_models_error: {row.n_models_error}")
        print(f"models_error: {row.models_error}")
        print(f"embedding_models_error: {row.embedding_models_error}")
        print(f"classifiers_error: {row.classifiers_error}")

print("\n" + "=" * 100)
print("Resumo final:")
print(df_top2[[
    "dataset_id",
    "idx",
    "true_label_id",
    "error_count",
    "n_models_error"
]])

dataset_id: CENTRALFATOS

Top 1
idx: CENTRALFATOS_review_03179
true_label_id: 1
error_count: 32
n_models_error: 32
models_error: ['albertina_ptbr_100m__decision_tree', 'albertina_ptbr_100m__gaussian_nb', 'albertina_ptbr_100m__gradient_boosting', 'albertina_ptbr_100m__knn', 'albertina_ptbr_100m__linearsvc', 'albertina_ptbr_100m__logreg', 'albertina_ptbr_100m__random_forest', 'albertina_ptbr_100m__svc_rbf', 'albertina_ptbr_900m__decision_tree', 'albertina_ptbr_900m__gaussian_nb', 'albertina_ptbr_900m__gradient_boosting', 'albertina_ptbr_900m__knn', 'albertina_ptbr_900m__linearsvc', 'albertina_ptbr_900m__logreg', 'albertina_ptbr_900m__random_forest', 'albertina_ptbr_900m__svc_rbf', 'bertimbau_base__decision_tree', 'bertimbau_base__gaussian_nb', 'bertimbau_base__gradient_boosting', 'bertimbau_base__knn', 'bertimbau_base__linearsvc', 'bertimbau_base__logreg', 'bertimbau_base__random_forest', 'bertimbau_base__svc_rbf', 'bertimbau_large__decision_tree', 'bertimbau_large__gaussian_nb', 'bertim

In [ ]:
df_top2["idx"].unique()

<StringArray>
['CENTRALFATOS_review_03179', 'CENTRALFATOS_review_14397',
          'COVID19BR_000086',          'COVID19BR_000361',
           'FACTCKBR_000956',         'FACTCKBR_000921_1',
             'FAKEBR_000319',             'FAKEBR_000953',
         'FAKETRUEBR_003348',         'FAKETRUEBR_003407',
                'FCN_001074',                'FCN_001169',
           'FNEWSSET_000032',           'FNEWSSET_000258',
           'FRECOGNA_001793',           'FRECOGNA_000186',
            'MUMINPT_000349',            'MUMINPT_001403',
             'TRE300_000143',             'TRE300_000445']
Length: 20, dtype: str

In [ ]:
import pandas as pd 

dataset = pd.read_csv(r"\codes\datasets\Unified_PTBR_FakeNews.csv")
dataset['dataset_id'].unique()

<StringArray>
[   'COVID19BR',       'FAKEBR',     'FNEWSSET',     'FRECOGNA',
      'MUMINPT', 'CENTRALFATOS',          'FCN',       'TRE300',
   'FAKETRUEBR',     'FACTCKBR',     'BOATOSBR']
Length: 11, dtype: str

In [21]:
import pandas as pd

# garante texto válido
dataset['text_no_url'] = dataset['text_no_url'].fillna('').astype(str)

# contagem simples de tokens por espaço
dataset['n_tokens'] = dataset['text_no_url'].str.split().str.len()

resumo = (
    dataset
    .groupby(['dataset_id', 'label'])
    .agg(
        qtd_registros=('label', 'size'),
        media_tokens=('n_tokens', 'mean'),
        max_tokens=('n_tokens', 'max'),
        min_tokens=('n_tokens', 'min'),
        desvio_tokens=('n_tokens', 'std')
    )
    .reset_index()
)

# opcional: arredondar
resumo['media_tokens'] = resumo['media_tokens'].round(2)
resumo['desvio_tokens'] = resumo['desvio_tokens'].round(2)

resumo

,dataset_id,label,qtd_registros,media_tokens,max_tokens,min_tokens,desvio_tokens
0,BOATOSBR,FAKE,1888,135.13,1309,7,147.75
1,BOATOSBR,REAL,1516,611.94,5579,23,447.53
2,CENTRALFATOS,FAKE,10282,649.50,8079,15,389.87
3,CENTRALFATOS,REAL,34,373.21,1763,123,319.11
4,COVID19BR,FAKE,905,172.16,4859,2,290.98
5,COVID19BR,REAL,1494,104.85,5938,1,296.30
6,FACTCKBR,FAKE,1866,42.24,232,8,35.60
7,FACTCKBR,REAL,240,35.68,86,10,17.75
8,FAKEBR,FAKE,3600,182.75,2088,9,115.72
9,FAKEBR,REAL,3600,184.30,2095,10,116.06


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

base_dir = Path(r"codes\results\embeddings")

all_dfs = []

for model_dir in base_dir.iterdir():
    if not model_dir.is_dir():
        continue

    meta_path = model_dir / "test_metadata.csv"
    emb_path = model_dir / "test_embeddings.npy"

    if not meta_path.exists() or not emb_path.exists():
        print(f"[AVISO] Arquivos não encontrados para {model_dir.name}")
        continue

    print(f"[INFO] Processando: {model_dir.name}")

    df_meta = pd.read_csv(meta_path)
    emb = np.load(emb_path)

    if len(df_meta) != len(emb):
        print(f"[ERRO] Tamanho incompatível em {model_dir.name}: metadata={len(df_meta)} | embeddings={len(emb)}")
        continue

    df_emb = pd.DataFrame(
        emb,
        columns=[f"emb_{i}" for i in range(emb.shape[1])]
    )

    df_model = pd.concat(
        [df_meta.reset_index(drop=True), df_emb.reset_index(drop=True)],
        axis=1
    )

    # garante coluna com nome do modelo
    df_model["embedding_model"] = model_dir.name

    all_dfs.append(df_model)

df_final = pd.concat(all_dfs, ignore_index=True)

print(df_final.shape)
print(df_final["embedding_model"].value_counts())
df_final.head()

[INFO] Processando: albertina_ptbr_100m
[INFO] Processando: albertina_ptbr_900m
[INFO] Processando: bertimbau_base
[INFO] Processando: bertimbau_large
[AVISO] Arquivos não encontrados para umap_plots
(25124, 1541)
embedding_model
albertina_ptbr_100m    6281
albertina_ptbr_900m    6281
bertimbau_base         6281
bertimbau_large        6281
Name: count, dtype: int64


,idx,dataset_id,label_id,split,embedding_model,emb_0,emb_1,emb_2,emb_3,emb_4,...,emb_1526,emb_1527,emb_1528,emb_1529,emb_1530,emb_1531,emb_1532,emb_1533,emb_1534,emb_1535
0,FRECOGNA_004833,FRECOGNA,0,test,albertina_ptbr_100m,-0.380482,0.755458,0.212023,0.039621,0.427192,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CENTRALFATOS_review_19507,CENTRALFATOS,0,test,albertina_ptbr_100m,-0.154427,0.849030,0.185304,-0.020128,0.349633,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,FAKEBR_003539,FAKEBR,1,test,albertina_ptbr_100m,0.028575,1.045952,0.421991,0.029603,0.226797,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FCN_000362,FCN,0,test,albertina_ptbr_100m,0.091543,0.724567,0.216048,0.215647,0.397498,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,FRECOGNA_000620,FRECOGNA,0,test,albertina_ptbr_100m,-0.165926,0.454289,0.140404,0.466125,0.131432,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_final

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap.umap_ as umap

from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


# =========================================================
# CONFIG GERAL
# =========================================================
RANDOM_STATE = 42

BASE_EMBEDDINGS_DIR = Path(
    r"codes\results\embeddings"
)

OUTPUT_DIR = Path(
    r"codes\results\umap_visualizations"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_IDXS = [
    "CENTRALFATOS_review_03179", "CENTRALFATOS_review_14397",
    "COVID19BR_000086", "COVID19BR_000361",
    "FACTCKBR_000956", "FACTCKBR_000921_1",
    "FAKEBR_000319", "FAKEBR_000953",
    "FAKETRUEBR_003348", "FAKETRUEBR_003407",
    "FCN_001074", "FCN_001169",
    "FNEWSSET_000032", "FNEWSSET_000258",
    "FRECOGNA_001793", "FRECOGNA_000186",
    "MUMINPT_000349", "MUMINPT_001403",
    "TRE300_000143", "TRE300_000445",
]

MODEL_CONFIGS = {
    "bertimbau_base": {
        "hf_name": "neuralmind/bert-base-portuguese-cased",
    },
    "bertimbau_large": {
        "hf_name": "neuralmind/bert-large-portuguese-cased",
    },
    "albertina_ptbr_100m": {
        "hf_name": "PORTULAN/albertina-100m-portuguese-ptbr-encoder",
    },
    "albertina_ptbr_900m": {
        "hf_name": "PORTULAN/albertina-900m-portuguese-ptbr-encoder",
    },
}


# =========================================================
# LOAD DE TODOS OS EMBEDDINGS
# =========================================================
def load_all_embeddings(base_dir: Path, split: str = "test") -> pd.DataFrame:
    all_dfs = []

    for model_dir in sorted(base_dir.iterdir()):
        if not model_dir.is_dir():
            continue

        meta_path = model_dir / f"{split}_metadata.csv"
        emb_path = model_dir / f"{split}_embeddings.npy"

        if not meta_path.exists() or not emb_path.exists():
            print(f"[AVISO] Arquivos não encontrados para {model_dir.name} ({split})")
            continue

        print(f"[INFO] Carregando {model_dir.name} | split={split}")

        df_meta = pd.read_csv(meta_path)
        emb = np.load(emb_path, allow_pickle=False)

        if emb.ndim != 2:
            print(f"[ERRO] Embeddings inválidos em {model_dir.name}: shape={emb.shape}")
            continue

        if len(df_meta) != emb.shape[0]:
            print(
                f"[ERRO] Tamanho incompatível em {model_dir.name}: "
                f"metadata={len(df_meta)} | embeddings={emb.shape[0]}"
            )
            continue

        df_model = df_meta.reset_index(drop=True).copy()
        df_model["embedding_model"] = model_dir.name
        df_model["embedding_dim"] = emb.shape[1]
        df_model["split"] = split

        # cada linha guarda o vetor completo
        df_model["embedding_vector"] = [row.astype(np.float32) for row in emb]

        all_dfs.append(df_model)

    if not all_dfs:
        raise ValueError("Nenhum embedding válido foi carregado.")

    df_final = pd.concat(all_dfs, ignore_index=True)

    print("\n[OK] df_final criado com sucesso")
    print(df_final.shape)
    print("\n[INFO] Modelos carregados:")
    print(
        df_final[["embedding_model", "embedding_dim"]]
        .drop_duplicates()
        .sort_values(["embedding_model"])
        .to_string(index=False)
    )

    return df_final


# =========================================================
# HELPERS
# =========================================================
def normalize_idx(x: str) -> str:
    x = str(x).strip()
    x = x.replace("CENTRALFATOS_review_", "CENTRALFATOS_")
    return x


def prepare_embeddings_matrix(df_model: pd.DataFrame, pca_components: int = 50) -> np.ndarray:
    X = np.vstack(df_model["embedding_vector"].values).astype(np.float32)

    # normalização L2
    X = normalize(X, norm="l2")

    # PCA antes do UMAP
    max_pca = min(X.shape[0] - 1, X.shape[1], pca_components)
    if max_pca >= 2 and X.shape[1] > max_pca:
        pca = PCA(n_components=max_pca, random_state=RANDOM_STATE)
        X = pca.fit_transform(X)

    return X


def infer_model_family(model_name: str) -> str:
    name = model_name.lower()
    if "large" in name or "900m" in name:
        return "large"
    return "base"


def build_umap_presets(model_name: str, n_samples: int):
    family = infer_model_family(model_name)

    if family == "large":
        base_neighbors = 40
    else:
        base_neighbors = 25

    max_neighbors = max(2, n_samples - 1)
    base_neighbors = min(base_neighbors, max_neighbors)
    local_neighbors = min(max(10, base_neighbors // 2), max_neighbors)
    global_neighbors = min(max(base_neighbors * 2, base_neighbors + 10), max_neighbors)

    return {
        "2d_local": {
            "n_components": 2,
            "n_neighbors": local_neighbors,
            "min_dist": 0.00,
            "metric": "cosine",
            "spread": 1.0,
        },
        "2d_balanced": {
            "n_components": 2,
            "n_neighbors": base_neighbors,
            "min_dist": 0.10,
            "metric": "cosine",
            "spread": 1.0,
        },
        "2d_global": {
            "n_components": 2,
            "n_neighbors": global_neighbors,
            "min_dist": 0.30,
            "metric": "cosine",
            "spread": 1.2,
        },
        "3d_balanced": {
            "n_components": 3,
            "n_neighbors": base_neighbors,
            "min_dist": 0.12,
            "metric": "cosine",
            "spread": 1.0,
        },
    }


def run_umap(X: np.ndarray, params: dict) -> np.ndarray:
    reducer = umap.UMAP(
        n_components=params["n_components"],
        n_neighbors=params["n_neighbors"],
        min_dist=params["min_dist"],
        metric=params["metric"],
        spread=params.get("spread", 1.0),
        random_state=RANDOM_STATE,
        transform_seed=RANDOM_STATE,
    )
    return reducer.fit_transform(X)


def build_masks(df_plot: pd.DataFrame):
    mask_label0 = df_plot["label_id"] == 0
    mask_label1 = df_plot["label_id"] == 1
    mask_target = df_plot["is_target"]

    return {
        "bg_0": mask_label0 & (~mask_target),
        "bg_1": mask_label1 & (~mask_target),
        "tg_0": mask_label0 & mask_target,
        "tg_1": mask_label1 & mask_target,
    }


# =========================================================
# PLOTS
# =========================================================
def plot_umap_2d(df_plot: pd.DataFrame, coords: np.ndarray, title: str, save_path: Path):
    df_aux = df_plot.copy()
    df_aux["u1"] = coords[:, 0]
    df_aux["u2"] = coords[:, 1]

    masks = build_masks(df_aux)

    plt.figure(figsize=(12, 9))

    plt.scatter(
        df_aux.loc[masks["bg_0"], "u1"],
        df_aux.loc[masks["bg_0"], "u2"],
        s=20, alpha=0.30, c="#9ecae1", label="Test - Label 0"
    )
    plt.scatter(
        df_aux.loc[masks["bg_1"], "u1"],
        df_aux.loc[masks["bg_1"], "u2"],
        s=20, alpha=0.30, c="#fcbba1", label="Test - Label 1"
    )
    plt.scatter(
        df_aux.loc[masks["tg_0"], "u1"],
        df_aux.loc[masks["tg_0"], "u2"],
        s=120, alpha=0.95, c="#08519c",
        edgecolors="black", linewidths=0.9,
        label="Highlighted - Label 0"
    )
    plt.scatter(
        df_aux.loc[masks["tg_1"], "u1"],
        df_aux.loc[masks["tg_1"], "u2"],
        s=120, alpha=0.95, c="#cb181d",
        edgecolors="black", linewidths=0.9,
        label="Highlighted - Label 1"
    )

    df_targets = df_aux[df_aux["is_target"]].copy()
    for _, row in df_targets.iterrows():
        plt.annotate(
            row["idx"],
            (row["u1"], row["u2"]),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
            alpha=0.95
        )

    plt.title(title, fontsize=14)
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_umap_3d(df_plot: pd.DataFrame, coords: np.ndarray, title: str, save_path: Path):
    df_aux = df_plot.copy()
    df_aux["u1"] = coords[:, 0]
    df_aux["u2"] = coords[:, 1]
    df_aux["u3"] = coords[:, 2]

    masks = build_masks(df_aux)

    fig = plt.figure(figsize=(13, 10))
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(
        df_aux.loc[masks["bg_0"], "u1"],
        df_aux.loc[masks["bg_0"], "u2"],
        df_aux.loc[masks["bg_0"], "u3"],
        s=18, alpha=0.18, c="#9ecae1", label="Test - Label 0"
    )
    ax.scatter(
        df_aux.loc[masks["bg_1"], "u1"],
        df_aux.loc[masks["bg_1"], "u2"],
        df_aux.loc[masks["bg_1"], "u3"],
        s=18, alpha=0.18, c="#fcbba1", label="Test - Label 1"
    )
    ax.scatter(
        df_aux.loc[masks["tg_0"], "u1"],
        df_aux.loc[masks["tg_0"], "u2"],
        df_aux.loc[masks["tg_0"], "u3"],
        s=120, alpha=0.95, c="#08519c",
        edgecolors="black", linewidths=0.8,
        label="Highlighted - Label 0"
    )
    ax.scatter(
        df_aux.loc[masks["tg_1"], "u1"],
        df_aux.loc[masks["tg_1"], "u2"],
        df_aux.loc[masks["tg_1"], "u3"],
        s=120, alpha=0.95, c="#cb181d",
        edgecolors="black", linewidths=0.8,
        label="Highlighted - Label 1"
    )

    for _, row in df_aux[df_aux["is_target"]].iterrows():
        ax.text(row["u1"], row["u2"], row["u3"], row["idx"], fontsize=7)

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.set_zlabel("UMAP-3")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_summary_figure(df_plot: pd.DataFrame, coords_dict: dict, model_name: str, save_path: Path):
    fig = plt.figure(figsize=(22, 16))

    titles = [
        ("2d_local", "2D Local"),
        ("2d_balanced", "2D Balanced"),
        ("2d_global", "2D Global"),
    ]

    for i, (key, subtitle) in enumerate(titles, start=1):
        ax = fig.add_subplot(2, 2, i)

        coords = coords_dict[key]
        df_aux = df_plot.copy()
        df_aux["u1"] = coords[:, 0]
        df_aux["u2"] = coords[:, 1]

        masks = build_masks(df_aux)

        ax.scatter(df_aux.loc[masks["bg_0"], "u1"], df_aux.loc[masks["bg_0"], "u2"],
                   s=18, alpha=0.28, c="#9ecae1", label="Test - Label 0")
        ax.scatter(df_aux.loc[masks["bg_1"], "u1"], df_aux.loc[masks["bg_1"], "u2"],
                   s=18, alpha=0.28, c="#fcbba1", label="Test - Label 1")
        ax.scatter(df_aux.loc[masks["tg_0"], "u1"], df_aux.loc[masks["tg_0"], "u2"],
                   s=100, alpha=0.95, c="#08519c",
                   edgecolors="black", linewidths=0.8, label="Highlighted - Label 0")
        ax.scatter(df_aux.loc[masks["tg_1"], "u1"], df_aux.loc[masks["tg_1"], "u2"],
                   s=100, alpha=0.95, c="#cb181d",
                   edgecolors="black", linewidths=0.8, label="Highlighted - Label 1")

        for _, row in df_aux[df_aux["is_target"]].iterrows():
            ax.annotate(
                row["idx"],
                (row["u1"], row["u2"]),
                xytext=(3, 3),
                textcoords="offset points",
                fontsize=7,
                alpha=0.90
            )

        ax.set_title(f"{model_name} - {subtitle}")
        ax.set_xlabel("UMAP-1")
        ax.set_ylabel("UMAP-2")
        ax.legend(fontsize=8)

    ax4 = fig.add_subplot(2, 2, 4, projection="3d")
    coords3d = coords_dict["3d_balanced"]

    df_aux = df_plot.copy()
    df_aux["u1"] = coords3d[:, 0]
    df_aux["u2"] = coords3d[:, 1]
    df_aux["u3"] = coords3d[:, 2]

    masks = build_masks(df_aux)

    ax4.scatter(df_aux.loc[masks["bg_0"], "u1"], df_aux.loc[masks["bg_0"], "u2"], df_aux.loc[masks["bg_0"], "u3"],
                s=16, alpha=0.16, c="#9ecae1", label="Test - Label 0")
    ax4.scatter(df_aux.loc[masks["bg_1"], "u1"], df_aux.loc[masks["bg_1"], "u2"], df_aux.loc[masks["bg_1"], "u3"],
                s=16, alpha=0.16, c="#fcbba1", label="Test - Label 1")
    ax4.scatter(df_aux.loc[masks["tg_0"], "u1"], df_aux.loc[masks["tg_0"], "u2"], df_aux.loc[masks["tg_0"], "u3"],
                s=100, alpha=0.95, c="#08519c",
                edgecolors="black", linewidths=0.8, label="Highlighted - Label 0")
    ax4.scatter(df_aux.loc[masks["tg_1"], "u1"], df_aux.loc[masks["tg_1"], "u2"], df_aux.loc[masks["tg_1"], "u3"],
                s=100, alpha=0.95, c="#cb181d",
                edgecolors="black", linewidths=0.8, label="Highlighted - Label 1")

    for _, row in df_aux[df_aux["is_target"]].iterrows():
        ax4.text(row["u1"], row["u2"], row["u3"], row["idx"], fontsize=6)

    ax4.set_title(f"{model_name} - 3D Balanced")
    ax4.set_xlabel("UMAP-1")
    ax4.set_ylabel("UMAP-2")
    ax4.set_zlabel("UMAP-3")
    ax4.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


# =========================================================
# PIPELINE PRINCIPAL
# =========================================================
def generate_umap_visualizations(df_final: pd.DataFrame):
    df = df_final.copy()

    df["split"] = df["split"].astype(str).str.strip().str.lower()
    df = df[df["split"] == "test"].copy()

    if df.empty:
        raise ValueError("Não há dados no split test.")

    df["idx_norm"] = df["idx"].astype(str).apply(normalize_idx)
    target_idxs_norm = {normalize_idx(x) for x in TARGET_IDXS}
    df["is_target"] = df["idx_norm"].isin(target_idxs_norm)

    df["label_id"] = pd.to_numeric(df["label_id"], errors="coerce")
    df = df.dropna(subset=["label_id", "embedding_vector"]).copy()
    df["label_id"] = df["label_id"].astype(int)

    model_names = sorted(df["embedding_model"].dropna().unique().tolist())

    for model_name in model_names:
        print("=" * 100)
        print(f"[INFO] Processing embedding_model: {model_name}")

        df_model = df[df["embedding_model"] == model_name].copy()

        if df_model.empty:
            print(f"[WARNING] No rows found for {model_name}")
            continue

        print(f"[INFO] n_samples={len(df_model)} | embedding_dim={df_model['embedding_dim'].iloc[0]}")

        X = prepare_embeddings_matrix(df_model, pca_components=50)
        presets = build_umap_presets(model_name, n_samples=len(df_model))

        model_output_dir = OUTPUT_DIR / model_name
        model_output_dir.mkdir(parents=True, exist_ok=True)

        coords_dict = {}

        for preset_name, params in presets.items():
            print(f"   -> Running {preset_name} with params={params}")
            coords = run_umap(X, params)
            coords_dict[preset_name] = coords

            save_path = model_output_dir / f"{model_name}_{preset_name}.png"

            if params["n_components"] == 2:
                plot_umap_2d(
                    df_plot=df_model,
                    coords=coords,
                    title=f"{model_name} - {preset_name.replace('_', ' ').title()}",
                    save_path=save_path
                )
            else:
                plot_umap_3d(
                    df_plot=df_model,
                    coords=coords,
                    title=f"{model_name} - {preset_name.replace('_', ' ').title()}",
                    save_path=save_path
                )

        summary_path = model_output_dir / f"{model_name}_summary.png"
        plot_summary_figure(
            df_plot=df_model,
            coords_dict=coords_dict,
            model_name=model_name,
            save_path=summary_path
        )

        print(f"[OK] Saved plots to: {model_output_dir}")


# =========================================================
# EXECUÇÃO
# =========================================================
# 1) carrega todos os embeddings de teste
df_final = load_all_embeddings(BASE_EMBEDDINGS_DIR, split="test")

# 2) gera UMAP para todos os modelos
generate_umap_visualizations(df_final)